# 01. 분석 – RFM · 4세그먼트 · 파레토 (코드정리 기준)

- 전처리_완료.csv 로드
- R·F만으로 4세그먼트: 충성(R≥4 & F≥4), 신규(R≥4 & F<4), 이탈위험(R<4 & F≥4), 이탈(R<4 & F<4)
- 파레토(고객 비율 vs 누적 매출), 상위 20% 내부 세그먼트 요약

In [1]:
import os
import pandas as pd
import numpy as np

# 전처리_완료.csv 경로: cwd·상대·상위 탐색 (실행 위치 무관)
_csv = None
_cwd = os.getcwd()
_candidates = [
    '전처리_완료.csv',
    os.path.join('데이터셋', '전처리_완료.csv'),
    os.path.join('..', '데이터셋', '전처리_완료.csv'),
    os.path.join('분석 과정', '전처리_완료.csv'),
    os.path.join('8조_리테일', '데이터셋', '전처리_완료.csv'),
    os.path.join('8조_리테일', '분석 과정', '전처리_완료.csv'),
]
for _p in _candidates:
    _abs = os.path.abspath(os.path.normpath(os.path.join(_cwd, _p)))
    if os.path.isfile(_abs):
        _csv = _abs
        break
if _csv is None:
    _d = _cwd
    for _ in range(6):
        _d = os.path.dirname(_d)
        if not _d: break
        for _sub in [os.path.join(_d, '8조_리테일', '데이터셋', '전처리_완료.csv'), os.path.join(_d, '데이터셋', '전처리_완료.csv')]:
            if os.path.isfile(_sub):
                _csv = _sub
                break
        if _csv: break
if _csv is None:
    raise FileNotFoundError(
        "전처리_완료.csv를 찾을 수 없습니다. 00_전처리_코드정리.ipynb를 먼저 실행한 뒤,"
        " 8조_리테일/데이터셋/ 또는 8조_리테일/분석 과정/ 에 파일이 있는지 확인하세요."
    )

df = pd.read_csv(_csv, encoding='utf-8-sig')
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['Sales'] = df['Quantity'] * df['UnitPrice']
print(f"로드: {len(df):,}행, 고객 {df['CustomerID'].nunique():,}명")

analysis_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)

rfm = (
    df.groupby('CustomerID')
    .agg(
        Recency=('InvoiceDate', lambda x: (analysis_date - x.max()).days),
        Frequency=('InvoiceNo', 'nunique'),
        Monetary=('Sales', 'sum')
    )
)
rfm['Monetary_log'] = np.log1p(rfm['Monetary'])

# R: 낮을수록 좋음 → 5,4,3,2,1
rfm['R_score'] = pd.qcut(rfm['Recency'], 5, labels=[5,4,3,2,1]).astype(int)
rfm['F_score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 5, labels=[1,2,3,4,5]).astype(int)
rfm['M_score'] = pd.qcut(rfm['Monetary'], 5, labels=[1,2,3,4,5]).astype(int)

def rfm_segment(row):
    if row['R_score'] >= 4 and row['F_score'] >= 4:
        return '충성 고객'
    elif row['R_score'] >= 4 and row['F_score'] < 4:
        return '신규 고객'
    elif row['R_score'] < 4 and row['F_score'] >= 4:
        return '이탈 위험'
    else:
        return '이탈 고객'

rfm['Segment'] = rfm.apply(rfm_segment, axis=1)
print('\n세그먼트별 고객 수:')
print(rfm['Segment'].value_counts())

로드: 393,515행, 고객 4,325명

세그먼트별 고객 수:
Segment
이탈 고객    1961
충성 고객    1131
신규 고객     634
이탈 위험     599
Name: count, dtype: int64


In [2]:
# R·F·M 점수별 구간
r_boundary = rfm.groupby('R_score')['Recency'].agg(['min', 'max']).sort_index(ascending=False)
f_boundary = rfm.groupby('F_score')['Frequency'].agg(['min', 'max']).sort_index()
m_boundary = rfm.groupby('M_score')['Monetary'].agg(['min', 'max']).sort_index()
print('--- Recency 점수별 구간 ---')
print(r_boundary)
print('\n--- Frequency 점수별 구간 ---')
print(f_boundary)
print('\n--- Monetary 점수별 구간 ---')
print(m_boundary)

--- Recency 점수별 구간 ---
         min  max
R_score          
5          1   14
4         15   33
3         34   72
2         73  180
1        181  374

--- Frequency 점수별 구간 ---
         min  max
F_score          
1          1    1
2          1    2
3          2    3
4          3    5
5          5  203

--- Monetary 점수별 구간 ---
             min        max
M_score                    
1           2.90     245.94
2         245.96     480.71
3         481.75     918.10
4         918.20    2013.22
5        2015.17  279522.50


In [3]:
# 파레토: 고객 비율 vs 누적 매출
customer_sales = (
    df.groupby('CustomerID')['Sales'].sum()
    .reset_index().sort_values('Sales', ascending=False).reset_index(drop=True)
)
customer_sales['cum_sales'] = customer_sales['Sales'].cumsum()
total_sales = customer_sales['Sales'].sum()
customer_sales['cum_ratio'] = customer_sales['cum_sales'] / total_sales
customer_sales['customer_ratio'] = (customer_sales.index + 1) / len(customer_sales)

idx_20 = (customer_sales['customer_ratio'] >= 0.20).idxmax()
x_20 = customer_sales.loc[idx_20, 'customer_ratio']
y_20 = customer_sales.loc[idx_20, 'cum_ratio']
idx_80 = (customer_sales['cum_ratio'] >= 0.80).idxmax()
x_80 = customer_sales.loc[idx_80, 'customer_ratio']
y_80 = customer_sales.loc[idx_80, 'cum_ratio']

print(f'고객 20% 지점 → 누적 매출 비중: {y_20*100:.1f}%')
print(f'매출 80% 지점 → 고객 비율: {x_80*100:.1f}%')

고객 20% 지점 → 누적 매출 비중: 73.6%
매출 80% 지점 → 고객 비율: 27.1%


In [4]:
# 상위 20% 고객 내부 세그먼트 요약 (코드정리: threshold 0.735 등)
threshold = 0.735
top_customers = customer_sales[customer_sales['cum_ratio'] <= threshold]
top_customers = top_customers.merge(rfm[['Segment']], left_on='CustomerID', right_index=True, how='left')

segment_sales = top_customers.groupby('Segment')['Sales'].sum().reset_index(name='Sales')
segment_sales['sales_ratio'] = segment_sales['Sales'] / segment_sales['Sales'].sum()
segment_count = top_customers.groupby('Segment')['CustomerID'].nunique().reset_index(name='CustomerCount')
segment_count['customer_ratio'] = segment_count['CustomerCount'] / segment_count['CustomerCount'].sum()
segment_summary_top = segment_sales.merge(segment_count, on='Segment')

print('상위 고객(누적 매출 73.5% 기준) 내 세그먼트 요약:')
print(segment_summary_top)

# 전체 고객 기준 세그먼트 요약
segment_sales_all = customer_sales.merge(rfm[['Segment']], left_on='CustomerID', right_index=True, how='left')
segment_sales_all = segment_sales_all.groupby('Segment')['Sales'].sum().reset_index(name='Sales')
segment_sales_all['sales_ratio'] = segment_sales_all['Sales'] / segment_sales_all['Sales'].sum()
segment_count_all = customer_sales.merge(rfm[['Segment']], left_on='CustomerID', right_index=True, how='left')
segment_count_all = segment_count_all.groupby('Segment')['CustomerID'].nunique().reset_index(name='CustomerCount')
segment_count_all['customer_ratio'] = segment_count_all['CustomerCount'] / segment_count_all['CustomerCount'].sum()
segment_summary_all = segment_sales_all.merge(segment_count_all, on='Segment')
print('\n전체 고객 기준 세그먼트 요약:')
print(segment_summary_all)

상위 고객(누적 매출 73.5% 기준) 내 세그먼트 요약:
  Segment       Sales  sales_ratio  CustomerCount  customer_ratio
0   신규 고객    74336.29     0.012111             20        0.023283
1   이탈 고객   194008.31     0.031608             52        0.060536
2   이탈 위험   728904.93     0.118753            182        0.211874
3   충성 고객  5140740.80     0.837528            605        0.704307

전체 고객 기준 세그먼트 요약:
  Segment       Sales  sales_ratio  CustomerCount  customer_ratio
0   신규 고객   394519.34     0.047238            634        0.146590
1   이탈 고객  1040461.29     0.124579           1961        0.453410
2   이탈 위험  1157259.63     0.138564            599        0.138497
3   충성 고객  5759579.06     0.689620           1131        0.261503


## 핵심 요약

| 항목 | 결과 |
|------|------|
| **분석 대상** | 전처리_완료.csv 기준 고객 **4,325명** |
| **세그먼트** | R·F만 사용 4분할: 충성(R≥4 & F≥4), 신규(R≥4 & F<4), 이탈위험(R<4 & F≥4), 이탈(R<4 & F<4) |
| **파레토** | 상위 **20% 고객**이 누적 매출의 **73.5%** 차지 (고객 비율 vs 누적 매출 곡선으로 검증) |
| **상위 20% 내부** | 이탈위험+충성 = 고객 비중 91.7%, 매출 비중 95.7% → 이 두 그룹 마케팅 집중 권장 |